# 10 AI Incident Database: reported AI incidents by alleged developer

**Purpose.** This notebook reads the weekly Excel export of the AI Incident Database (AIID) and turns it into four tidy tables: one row per incident, one row per incident and alleged developer, a count of incidents per developer and year, and a count per year split into generative-AI related and other. Q6 uses it to show whether reported incidents are rising and which developers' systems appear in them.

**Question served.** Q6 (reported incidents). Evidence chapter, view V8.

**Important reading rules.** The `developer` field records who is **alleged** to have built the AI system involved. It is not a finding of fault. Incidents come from news reports, mostly in English, so counts reflect reporting as well as events. Developer names are free text. This notebook cleans them with a small hand-made rule list and keeps the original text next to the cleaned name.

**How to run.** Run all cells from top to bottom. The notebook tries to download the newest weekly export. If that is blocked, it uses the Excel file you saved by hand in `Data/raw/manual/`.

## At a glance: links, access, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one Excel file (the weekly export), read in memory |
| Publisher | Responsible AI Collaborative, AI Incident Database |
| Snapshots page | https://incidentdatabase.ai/research/snapshots/ |
| File used | https://pub-72b2b2fc36ec423189843747af98f80e.r2.dev/AIID_Excel_Export-YYYYMMDD.xlsx (a new file every Monday) |
| Access | Public file. No account, key or login needed. No scraping |
| Download size | about 2.6 MB |
| Expected time | download 2 to 10 seconds, reading the sheet 3 to 10 seconds, whole notebook under 30 seconds |
| Data as of | the date in the file name (snapshot day) and the latest incident date inside the file. See `processed/data_currency.csv` |
| To update | Run the notebook again. It tries the newest Monday first and then the five before it |

## 1. Source and reference links

| Item | Link |
|---|---|
| AIID home | https://incidentdatabase.ai |
| Snapshots and backups page | https://incidentdatabase.ai/research/snapshots/ |
| Excel export (pattern) | https://pub-72b2b2fc36ec423189843747af98f80e.r2.dev/AIID_Excel_Export-YYYYMMDD.xlsx |
| Latest file we hold | https://pub-72b2b2fc36ec423189843747af98f80e.r2.dev/AIID_Excel_Export-20261012.xlsx |
| Weekly database backup (not used here) | https://pub-72b2b2fc36ec423189843747af98f80e.r2.dev/backup-20261012101643.tar.bz2 |

**Citation.** McGregor, S. (2021). Preventing Repeated Real World AI Failures by Cataloging Incidents: The AI Incident Database. AAAI 2021. https://ojs.aaai.org/index.php/AAAI/article/view/17817

**How the export is organised.** The workbook has five sheets: `Incidents` (one row per incident, 59 columns), `Reports` (one row per linked news report), `Entities` (a list of entity names), `Data Dictionary` and `Coverage Map`. This notebook reads only `Incidents`. **Chosen time window.** The notebook keeps every year, and the chart in the app should start in 2022, the year ChatGPT launched, because before it generative-AI incidents were a small share and the models in the Epoch table were released from 2023.

## 2. Table structure

### 2.1 Source sheet `Incidents`

The first row is a title, the second row names column groups, and the third row holds the column names. The notebook finds the header row by looking for `Incident ID`, so a change in the number of title rows does not break it. Columns used:

| Source column | Meaning |
|---|---|
| `Incident ID` | unique number of the incident |
| `date`, `year` | date the harm occurred (editor-resolved) and its year |
| `title` | short title of the incident |
| `developer` | alleged developer or developers of the AI system, free text, several names separated by commas |
| `deployer` | alleged deployer, free text |
| `harmed` | parties harmed, free text |
| `report_count` | number of linked news reports |

### 2.2 Outputs in `processed/`

| File | Rows | Columns |
|---|---|---|
| `incidents.csv` | one per incident | `incident_id`, `date`, `year`, `title`, `developer_raw`, `deployer_raw`, `report_count`, `named_developers` (our cleaned names for known companies, joined with ` | `), `has_named_developer`, `is_generative_related` (our keyword rule, see Section 6) |
| `incident_developers.csv` | one per incident and developer name | `incident_id`, `date`, `year`, `developer_raw_entity`, `developer_clean`, `entity_kind` (`named_company`, `generic_category`, `unknown` or `other_entity`), `is_china_based` (our annotation), `genai_flag` |
| `incidents_by_developer_year.csv` | one per named developer and year | `developer_clean`, `year`, `incidents`, `is_china_based` |
| `incidents_by_year.csv` | one per year | `year`, `incidents`, `generative_related`, `other`, `generative_share_pct` |
| `data_currency.csv` | one | when extracted and what period the data covers |

## 3. Set up the paths and download helper

In [ ]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "10_aiid_incidents"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
elif (cwd / "Data" / FOLDER).exists():
    HERE = cwd / "Data" / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER} (or from Data/). Current folder: {cwd}")

DATA = HERE.parent               # the Data folder
MANUAL = DATA / "raw" / "manual"                # hand-downloaded or pre-copied source files
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)


HEADERS = {"User-Agent": "UTS-36104-A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 120
REFRESH = True   # True: try a fresh download first. False: reuse the file already in raw/ if there is one.


def record(source_id: str, url: str, path: Path, how: str) -> None:
    """Add (or replace) the retrieval-log entry for one raw file."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")
    print(f"logged {source_id}: {len(content):,} bytes via {how}")


def get_file(source_id: str, url: str, target: str, manual_name: str = None) -> Path:
    """Return the path of a raw file. Order: fresh download, earlier download in raw/, manual copy."""
    path = RAW / target
    if REFRESH or not path.exists():
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path.write_bytes(response.content)
            record(source_id, url, path, "downloaded")
            return path
        except Exception as err:
            print(f"Download of {source_id} failed: {err}")
    manual = MANUAL / (manual_name or target)
    candidates = [p for p in (path, manual) if p.exists()]
    if candidates:
        best = max(candidates, key=lambda p: p.stat().st_mtime)   # the newest copy wins
        if best == path:
            print(f"Using the earlier download {path.name}")
        else:
            shutil.copy2(best, path)
            record(source_id, url, path, f"copied from manual folder {manual.parent.name}")
        return path
    raise FileNotFoundError(
        f"Could not get {source_id}. Open {url} in a browser, save the file as '{manual_name or target}' "
        f"inside {MANUAL}, then run this cell again.")

import re
from datetime import timedelta

## 4. Acquire the newest weekly export

The export file name carries its date and a new file appears every Monday. The notebook tries the newest Monday first and then goes back up to five weeks. If every try fails, it uses the newest Excel file from `Data/raw/manual/`. A file lying loose in `Data/` is also accepted.

In [ ]:
BASE = "https://pub-72b2b2fc36ec423189843747af98f80e.r2.dev"
FILE_RE = re.compile(r"AIID_Excel_Export-(\d{8})\.xlsx$")

today = datetime.now(timezone.utc).date()
last_monday = today - timedelta(days=today.weekday())
downloaded = None
if REFRESH:
    for k in range(6):
        day = last_monday - timedelta(weeks=k)
        name = f"AIID_Excel_Export-{day:%Y%m%d}.xlsx"
        url = f"{BASE}/{name}"
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            if response.status_code == 200 and response.content[:2] == b"PK":
                (RAW / name).write_bytes(response.content)
                record("aiid_excel_export", url, RAW / name, "downloaded")
                downloaded = RAW / name
                break
            print(f"  {name}: HTTP {response.status_code}")
        except Exception as err:
            print(f"  {name}: {err}")
            break          # a network problem will not fix itself on an earlier week
    if downloaded is None:
        print("No fresh download. Looking for a file saved by hand or downloaded earlier.")

pool = list(RAW.glob("AIID_Excel_Export-*.xlsx")) + list(MANUAL.glob("AIID_Excel_Export-*.xlsx")) + list(DATA.glob("AIID_Excel_Export-*.xlsx"))
pool = [p for p in pool if FILE_RE.search(p.name)]
if not pool:
    raise FileNotFoundError(
        "No AIID export found. Open https://incidentdatabase.ai/research/snapshots/ in a browser, "
        f"download the newest AIID_Excel_Export-YYYYMMDD.xlsx and save it in {MANUAL}, then run this cell again.")
best = max(pool, key=lambda p: FILE_RE.search(p.name).group(1))     # newest snapshot date wins
if best.parent != RAW:
    shutil.copy2(best, RAW / best.name)
    record("aiid_excel_export", f"{BASE}/{best.name}", RAW / best.name, f"copied from {best.parent.name}")
XLSX = RAW / best.name
SNAPSHOT = datetime.strptime(FILE_RE.search(XLSX.name).group(1), "%Y%m%d").date()
print("Using", XLSX.name, "snapshot day", SNAPSHOT)

## 5. Read the `Incidents` sheet

In [ ]:
probe = pd.read_excel(XLSX, sheet_name="Incidents", header=None, nrows=8)
header_row = next(i for i in range(len(probe)) if str(probe.iloc[i, 0]).strip() == "Incident ID")
print("Header row index:", header_row)

KEEP = ["Incident ID", "date", "year", "title", "deployer", "developer", "harmed", "report_count"]
inc = pd.read_excel(XLSX, sheet_name="Incidents", header=header_row, usecols=lambda c: str(c).strip() in KEEP)
inc.columns = [str(c).strip() for c in inc.columns]
missing = [c for c in KEEP if c not in inc.columns]
assert not missing, f"Columns not found: {missing}. The export layout may have changed."
inc["date"] = pd.to_datetime(inc["date"], errors="coerce")
print(inc.shape)
inc.head(5)

## 6. Clean the developer names

The `developer` text holds one or several names separated by commas, for example `Openai, Large Language Model Developers, Chatbot Developers`. The notebook splits it into entities and sorts each one into a kind:

- **named_company** matches one of our rules below (a company we want to count by name).
- **generic_category** is a technology label such as "Deepfake Technology Developers". It is not a company.
- **unknown** is "Unknown" or similar.
- **other_entity** is anything else (other companies, public bodies, individuals). It is kept but not counted by name.

**Generative-AI flag.** An incident is flagged `is_generative_related` when any of its developer entities contains a keyword such as language model, generative, chatbot, deepfake, synthetic or image generation, or is one of the model makers OpenAI, Anthropic, xAI, Mistral, DeepSeek, Moonshot, MiniMax or Z.ai. It is a rough rule on the developer text only and can miss incidents whose developer is a company like Google or Meta.

The rule list, the generative flag and the China-based flag are **our annotation**. Add a line to `NAMED` to count another company.

In [ ]:
NAMED = [   # (clean name, regex on the lower-case entity text, is_china_based)
    ("OpenAI",     r"\bopenai\b", False),
    ("Google",     r"\b(google|deepmind|alphabet)\b", False),
    ("Meta",       r"^(meta|facebook|meta platforms|meta ai)\b", False),
    ("Microsoft",  r"\bmicrosoft\b", False),
    ("Anthropic",  r"\banthropic\b", False),
    ("xAI",        r"^(xai|x\.ai)\b", False),
    ("Amazon",     r"\bamazon\b", False),
    ("Apple",      r"^apple\b", False),
    ("Tesla",      r"\btesla\b", False),
    ("Nvidia",     r"\bnvidia\b", False),
    ("Mistral",    r"\bmistral\b", False),
    ("DeepSeek",   r"deepseek", True),
    ("Alibaba",    r"alibaba|\bqwen\b|tongyi", True),
    ("Z.ai (Zhipu)", r"zhipu|\bz\.ai\b|chatglm", True),
    ("Moonshot",   r"moonshot|\bkimi\b", True),
    ("MiniMax",    r"minimax", True),
    ("Baidu",      r"baidu|\bernie\b", True),
    ("Tencent",    r"tencent|wechat", True),
    ("ByteDance",  r"bytedance|tiktok|douyin", True),
]
COMPILED = [(name, re.compile(rx), china) for name, rx, china in NAMED]


GENAI_WORDS = re.compile(r"large language|generative|chatbot|deepfake|synthetic|language model|image generation|llm", re.I)
GENAI_NAMED = {"OpenAI", "Anthropic", "xAI", "Mistral", "DeepSeek", "Moonshot", "MiniMax", "Z.ai (Zhipu)"}


def classify(entity):
    low = entity.lower().strip()
    for name, rx, china in COMPILED:
        if rx.search(low):
            return name, "named_company", china
    if re.search(r"\bdevelopers?$|\bcompanies$|technology$", low):
        return entity, "generic_category", False
    if low in ("unknown", "unnamed", "none", "") or low.startswith("unknown"):
        return entity, "unknown", False
    return entity, "other_entity", False


rows = []
for _, r in inc.iterrows():
    raw_text = r["developer"] if pd.notna(r["developer"]) else ""
    entities = [e.strip() for e in str(raw_text).split(",") if e.strip()] or ["Unknown"]
    for entity in entities:
        clean, kind, china = classify(entity)
        rows.append({"incident_id": r["Incident ID"], "date": r["date"], "year": r["year"],
                     "developer_raw_entity": entity, "developer_clean": clean,
                     "entity_kind": kind, "is_china_based": china,
                     "genai_flag": bool(GENAI_WORDS.search(entity)) or clean in GENAI_NAMED})
dev = pd.DataFrame(rows)
print(len(dev), "incident-developer rows from", dev["incident_id"].nunique(), "incidents")
print(dev["entity_kind"].value_counts().to_string())

## 7. Build the three tables

In [ ]:
named = dev[dev["entity_kind"].eq("named_company")]
per_incident = (named.drop_duplicates(["incident_id", "developer_clean"])
                .groupby("incident_id")["developer_clean"].agg(" | ".join))

incidents = pd.DataFrame({
    "incident_id": inc["Incident ID"],
    "date": inc["date"].dt.date,
    "year": inc["year"],
    "title": inc["title"],
    "developer_raw": inc["developer"],
    "deployer_raw": inc["deployer"],
    "report_count": inc["report_count"],
})
incidents["named_developers"] = incidents["incident_id"].map(per_incident).fillna("")
incidents["is_generative_related"] = incidents["incident_id"].map(dev.groupby("incident_id")["genai_flag"].any()).fillna(False)
incidents["has_named_developer"] = incidents["named_developers"].ne("")

by_year = (named.drop_duplicates(["incident_id", "developer_clean"])
           .groupby(["developer_clean", "year", "is_china_based"]).size()
           .reset_index(name="incidents")
           .sort_values(["developer_clean", "year"]).reset_index(drop=True))
per_year = (incidents.groupby("year").agg(incidents=("incident_id", "size"), generative_related=("is_generative_related", "sum"))
            .reset_index())
per_year["other"] = per_year["incidents"] - per_year["generative_related"]
per_year["generative_share_pct"] = (per_year["generative_related"] / per_year["incidents"] * 100).round(0)
print(incidents.shape, dev.shape, by_year.shape)
incidents.head(5)

## 8. Validate

In [ ]:
problems = []
if len(incidents) < 1500:
    problems.append(f"only {len(incidents)} incidents, the export may be truncated")
if incidents["incident_id"].duplicated().any():
    problems.append("duplicated incident ids")
if inc["date"].notna().mean() < 0.99:
    problems.append("more than 1% of the dates could not be read")
if inc["developer"].notna().mean() < 0.95:
    problems.append("developer is empty for more than 5% of the incidents")
agree = (inc["date"].dt.year == inc["year"]).mean()
if agree < 0.99:
    problems.append(f"year and date agree for only {agree:.1%} of the rows")
if dev["incident_id"].nunique() != len(incidents):
    problems.append("some incidents have no developer row")
if int(by_year["incidents"].sum()) != len(named.drop_duplicates(["incident_id", "developer_clean"])):
    problems.append("yearly counts do not add up to the named-developer rows")
if not (by_year["developer_clean"] == "OpenAI").any():
    problems.append("no incident matched OpenAI, the name rules or the layout may have changed")
print("Problems:", problems if problems else "none")
assert not problems, problems
print()
print("Incidents:", len(incidents), " date range:", inc["date"].min().date(), "to", inc["date"].max().date())
print("Incidents with at least one named company:", int(incidents["has_named_developer"].sum()))

## 9. Save

In [ ]:
incidents.to_csv(OUT / "incidents.csv", index=False, encoding="utf-8")
dev.assign(date=dev["date"].dt.date).to_csv(OUT / "incident_developers.csv", index=False, encoding="utf-8")
by_year.to_csv(OUT / "incidents_by_developer_year.csv", index=False, encoding="utf-8")
per_year.to_csv(OUT / "incidents_by_year.csv", index=False, encoding="utf-8")
print("saved 4 tables:", incidents.shape, dev.shape, by_year.shape, per_year.shape)

## 10. Quick look: how many incidents, and for whom

The first table shows incidents per year. The second counts incidents per named developer since 2024. The third compares developers flagged as China-based with the rest. China-based developers have very few records, so the comparison is shown only to make that limit visible.

In [ ]:
print("Incidents per year, and how many are generative-AI related by our keyword rule (2022 is the year ChatGPT launched):")
print(per_year[per_year["year"] >= 2018].to_string(index=False))
print()
recent = named[named["year"] >= 2024].drop_duplicates(["incident_id", "developer_clean"])
print("Incidents since 2024 per named developer (top 12):")
print(recent["developer_clean"].value_counts().head(12).to_string())
print()
cn = named.drop_duplicates(["incident_id", "developer_clean"])
print("All years, named developers flagged China-based vs not:")
print(cn.groupby("is_china_based")["incident_id"].nunique().rename({True: "China-based", False: "other named"}).to_string())

## How to update, and what to do if the download is blocked

**Automatic.** Run all cells again. Section 4 tries the newest Monday file and goes back up to five weeks. About 10 seconds.

**Manual download (use this if you see `No fresh download`).**

1. Open https://incidentdatabase.ai/research/snapshots/ in a browser.
2. Find the section for the Excel export and download the newest `AIID_Excel_Export-YYYYMMDD.xlsx`. Or paste the direct address, for example https://pub-72b2b2fc36ec423189843747af98f80e.r2.dev/AIID_Excel_Export-20261012.xlsx , into the address bar and change the date to the latest Monday.
3. Save it in `Data\raw\manual\` with the original name. Do not rename it, because the date in the name is how the notebook picks the newest file.
4. Run the notebook again. It uses the file with the newest date in its name.

**Update check.** Open `raw/retrieval_log.json` and read the `how` field (`downloaded` or `copied from manual`). Then open `processed/data_currency.csv` and confirm `data_covers_to` and the snapshot note.

**If the layout changes.** The run stops with a message that names the missing column or the count that failed. Open the `Data Dictionary` sheet of the workbook to see the new names.

## 11. Run time and limits

**Time to extract.** One request for a file of about 2.6 MB and one sheet read of about 1,700 rows by 59 columns. Expect 2 to 10 seconds for the download and 3 to 10 seconds for reading, so a full run takes **under 30 seconds**. The last cell prints the measured time.

**Limits and known risks.**

1. Incidents are reported events, mostly from English-language news. A higher count can mean more reporting and not more harm.
2. The developer is **alleged**. A developer name appearing in a row is not a finding of fault.
3. Developer text is free text and often generic ("Deepfake Technology Developers"). Many incidents have no named company at all. Check `has_named_developer` before quoting a share.
4. The four China-based developers in our scope (DeepSeek, Alibaba, Z.ai, Moonshot) appear in only 8 incidents over all years in the 12 October 2026 export (Alibaba 4, DeepSeek 3, Z.ai 1, Moonshot 0), against 190 for OpenAI. The data cannot support a comparison of risk between regions.
5. The name rules and the China-based flag are our annotation. They match on text and can miss a spelling variant or catch an unrelated entity. Print the rows of `other_entity` with the most incidents and extend `NAMED` if an important company is missing.
6. The generative-AI flag looks only at developer names, so it undercounts. Treat the share as a lower bound.
7. The most recent months are incomplete, because incidents are added after reports appear. Do not read a drop in the latest year as a fall in incidents.

## 12. References

1. AI Incident Database. https://incidentdatabase.ai
2. AI Incident Database, snapshots page. https://incidentdatabase.ai/research/snapshots/
3. McGregor, S. (2021). AAAI 2021. https://ojs.aaai.org/index.php/AAAI/article/view/17817

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [ ]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


write_currency("AI Incident Database export (incidents.csv)",
               inc["date"].min(), inc["date"].max(),
               f"weekly export dated {SNAPSHOT}. The latest incident date is the date the harm occurred and is usually earlier than the snapshot day, because incidents are added after reports appear",
               "Database start date not recorded in our notes. The earliest incident in the data is dated " + str(inc["date"].min().date()) + " (old incidents were added later). Check the home page for the project history.",
               "A new export every Monday")

In [ ]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")